In [ ]:
import duckdb 
import pandas as pd

In [ ]:
df = duckdb.sql('''
select * 
from "https://minio.lab.sspcloud.fr/severine/stock_RNE_formalites_20250523_0000.parquet"
where "individu.descriptionPersonne.nomUsage" IS NOT NULL and "entreprise.siren" IS NOT NULL
limit 10;
''').df ()
df

In [ ]:
df = duckdb.sql('''
select * 
from "https://minio.lab.sspcloud.fr/severine/stock_RNE_formalites_20250523_0000.parquet"
limit 10;
''').df ()
df

In [ ]:
df = duckdb.sql('''
select typePersonne, count(*)  
from "https://minio.lab.sspcloud.fr/severine/stock_RNE_formalites_20250523_0000.parquet"
group by typePersonne
''').df()
df

In [ ]:
df_base = pd.read_parquet("https://minio.lab.sspcloud.fr/severine/stock_RNE_formalites_20250523_0000.parquet") 
df_base.head()

In [ ]:
df_base = pd.read_parquet("https://minio.lab.sspcloud.fr/severine/stock_RNE_formalites_20250523_0000.parquet") 
df_base.columns

In [ ]:
# Vérification de la structure et des types de données des colonnes issues du fichier parquet

df_desc = duckdb.sql(f"""
DESCRIBE SELECT
    siren,
    nom,
    prenoms,
    nomUsage,
    dateDeNaissance,
    roleEntreprise,
    typePersonne
FROM read_parquet('{parquet_path}')
""").df()

df_desc

In [ ]:
df_base = pd.read_parquet("https://minio.lab.sspcloud.fr/severine/stock_RNE_formalites_20250523_0000.parquet") 
df_base.info()

In [ ]:
df_base = pd.read_parquet("https://minio.lab.sspcloud.fr/severine/stock_RNE_formalites_20250523_0000.parquet") 
df_base.isnull().sum()

In [ ]:
# Taux de remplissage des colonnes nom, prénoms, date de naissance

parquet_path = ("https://minio.lab.sspcloud.fr/severine/stock_RNE_formalites_20250523_0000.parquet")

# Lecture du parquet via DuckDB → conversion en DataFrame pandas
df = duckdb.sql(f"""
SELECT *
FROM read_parquet('{parquet_path}')
""").df()

# Taux de remplissage selon le type de personne

# Remplace chaîne vide par valeurs manquantes
df["nom"] = df["nom"].apply(
    lambda x: pd.NA if isinstance(x, str) and x.strip() == "" else x
)

# ------------------------------------------------------------
# Sous-ensembles
# ------------------------------------------------------------
df_physiques = df[df["typePersonne"] == "personnePhysique"].copy()
df_morales = df[df["typePersonne"] == "personneMorale"].copy()

total_physiques = len(df_physiques)
total_morales = len(df_morales)

# ------------------------------------------------------------
# PERSONNES PHYSIQUES
# Colonnes : nom, prenoms, dateDeNaissance
# ------------------------------------------------------------
resultats_physiques = pd.DataFrame({
    "nb_manquants": df_physiques[[
        "nom",
        "prenoms",
        "dateDeNaissance"
    ]].isna().sum()
})

resultats_physiques["nb_renseignes"] = total_physiques - resultats_physiques["nb_manquants"]

resultats_physiques["pourcentage_manquants"] = (
    resultats_physiques["nb_manquants"] / total_physiques * 100
).round(2)

resultats_physiques["pourcentage_remplissage"] = (
    resultats_physiques["nb_renseignes"] / total_physiques * 100
).round(2)

resultats_physiques["population"] = "personnePhysique"

# ------------------------------------------------------------
# PERSONNES MORALES
# Colonnes : individu.nom, individu.prenoms, individu.dateDeNaissance
# ------------------------------------------------------------
resultats_morales = pd.DataFrame({
    "nb_manquants": df_morales[[
        "individu.descriptionPersonne.nom",
        "individu.descriptionPersonne.prenoms",
        "individu.descriptionPersonne.dateDeNaissance"
    ]].isna().sum()
})

resultats_morales["nb_renseignes"] = total_morales - resultats_morales["nb_manquants"]

resultats_morales["pourcentage_manquants"] = (
    resultats_morales["nb_manquants"] / total_morales * 100
).round(2)

resultats_morales["pourcentage_remplissage"] = (
    resultats_morales["nb_renseignes"] / total_morales * 100
).round(2)

resultats_morales["population"] = "personneMorale"

# ------------------------------------------------------------
# Fusion des résultats
# ------------------------------------------------------------
resultats_final = pd.concat([resultats_physiques, resultats_morales])

# Mise en forme plus lisible
resultats_final = resultats_final.reset_index().rename(columns={"index": "colonne"})

# ------------------------------------------------------------
# Affichage
# ------------------------------------------------------------
print(f"Nombre total de personnes physiques : {total_physiques}")
print(f"Nombre total de personnes morales : {total_morales}")
resultats_final = resultats_final.reset_index().rename(columns={"index": "colonne"})
display(resultats_final)


**32% de date de naissance renseignée pour les personnes physiques**
**88% de date de naissance renseignée pour les personnes morales**


In [ ]:
df = duckdb.sql(f"""
SELECT *
FROM read_parquet('{parquet_path}')
LIMIT 50
""").df()

df